# Chop Block Bin
Build a GridFinity cutlery drawer organiser for holding IKEA-sized chopping blocks.

## 1. Imports, parameters, base

Import all the right commands. (I'm not using the recommended `from build123d import *` to improve my understanding of each command as I add it)
Set up some parameters to build the block.
Create and display the base for the bin, using the `gridfinity_build123d` package.

In [ ]:
from build123d import (
    MM,
    Align,
    Axis,
    BasePartObject,
    BaseSketchObject,
    BuildLine,
    BuildPart,
    BuildSketch,
    FilletPolyline,
    Line,
    Locations,
    Mode,
    Plane,
    Polyline,
    RectangleRounded,
    RotationLike,
    add,
    export_step,
    export_stl,
    extrude,
    make_face,
    mirror,
)
from gridfinity_build123d import BaseEqual, Bin
from ocp_vscode import set_port, show

# Set the desired port number
set_port(3939)

BASE_LENGTH = 6 # Units
BASE_WIDTH = 4 # Units
BASE_CORNER_RADIUS = 7.5 / 2 * MM # mm
HEIGHT = 63 * MM # mm

CHOP_LENGTH = 222 * MM # Length of chopping board in mm (plus 2mm for clearance)
CHOP_WIDTH  = 162 * MM # Width of chopping board in mm (plus 2mm for clearance)
CHOP_CORNER_RADIUS = 35 * MM # mm
CHOP_HEIGHT = (HEIGHT - 7) * MM # mm

SIDE_DOUBLE_LENGTH = 75 * MM # mm from outside edge of bin wall to edge of cutout
SIDE_HALF_LENGTH = (BASE_LENGTH * 42 * MM) / 2 # mm from centerline to outside edge of bin wall
CUTOUT_LENGTH = SIDE_HALF_LENGTH - SIDE_DOUBLE_LENGTH # mm from centerline to edge of cutout
CUTOUT_RADIUS = 12.5 * MM # mm radius of side cutout arc
CUTOUT_ARC = CUTOUT_LENGTH + CUTOUT_RADIUS + 0.1 * MM # mm from centerline to edge of cutout arc
CUTOUT_DEPTH = ((BASE_WIDTH * 42 * MM) - CHOP_WIDTH) / 2 # mm width of walls on long sides of chop block

BIN_BASE = BaseEqual(
    grid_x=BASE_WIDTH,
    grid_y=BASE_LENGTH,
)

BIN = Bin(
    base=BIN_BASE,
    height=HEIGHT,
)

# show(BIN_BASE)
show(BIN)

c


## 2. Scoop outline/profile

Creating the bin proved relatively straight-forward. Adding scoops so you can lift the chopping blocks out of the bin was more involved. This creates a "1D" outline for the scoop, turns it into a 2D profile and shows it. 

Using the first version of the profile to cut out the scoops from the side of the bin didn't work, but a conversation on Discord revealed this handy tip:
> Booleans (fusion, subtraction, intersection) sometimes struggle with coplanar geometry. Also, scoop_profile created a face that had some 0 width parts at the extremes of the top, that "arc+0.1" in X at the extremes was 0 in Y/Z, but this was not a problem when moving the sketch so 🤷‍♂️ . Moving the sketch 0.001 up is also something that can help in these cases.

Now the scoop outline has a 0.1 lintel, which solved the issue.

In [ ]:
with BuildSketch() as scoop_profile:
    with BuildLine() as scoop_outline:
        FilletPolyline(
            (0, 0),
            (CUTOUT_LENGTH, 0),
            (CUTOUT_LENGTH, CHOP_HEIGHT),
            (CUTOUT_ARC, CHOP_HEIGHT),
            radius=CUTOUT_RADIUS,
        )
        Polyline(
            (0, CHOP_HEIGHT + 0.1),
            (CUTOUT_ARC, CHOP_HEIGHT + 0.1),
            (CUTOUT_ARC, CHOP_HEIGHT),
        )

        mirror(about=Plane.YZ)

    make_face()

# show(scoop_outline)
show(scoop_profile)

-c


## 3. Scoop profile class

Converting the scoop profile into a class so it can be used in the future across different bins.

In [24]:
class ScoopProfile(BaseSketchObject):
    """Sketch Object: ScoopProfile

    Args:
        length (float): scoop length
        height (float): scoop height
        arc (float): arc length of scoop
        radius (float): radius of scoop arc
        rotation (float): angle to rotate about axes. Defaults to (0, 0, 0)
        align (Align | tuple[Align, Align, Align] | None, optional): align MIN, CENTER,
            or MAX of object. Defaults to (Align.CENTER, Align.CENTER, Align.CENTER)
        mode (Mode, optional): combine mode. Defaults to Mode.ADD
    """

    _applies_to = [BuildSketch._tag]

    def __init__(
        self,
        length: float,
        height: float,
        arc: float,
        radius: float,
        rotation: float = 0,
        align: tuple[Align, Align] = (Align.CENTER, Align.CENTER),
        mode: Mode = Mode.ADD,
    ):
        with BuildSketch() as scoop_profile:
            with BuildLine():
                FilletPolyline(
                    (0, 0),
                    (length, 0),
                    (length, height),
                    (arc, height),
                    radius=radius,
                )
                Polyline(
                    (0, height + 0.1),
                    (arc, height + 0.1),
                    (arc, height),
                )
                mirror(about=Plane.YZ)
            make_face()

        super().__init__(
            obj=scoop_profile.sketch,
            rotation=rotation,
            align=align,
            mode=mode
        )

test_scoop_profile = ScoopProfile(
    length=CUTOUT_LENGTH,
    height=CHOP_HEIGHT,
    arc=CUTOUT_ARC,
    radius=CUTOUT_RADIUS,
)
show(test_scoop_profile)

c

## 4. Chopping Block Bin

Create a GridFinity base and add a bin on top with the correct dimensions to hold an IKEA-sized chopping block. Show it.

In [ ]:
with BuildPart() as chop_part:
    # Add the base
    add(BIN)
    # Add a sketch on top for the chop compartment
    with BuildSketch(chop_part.faces().sort_by(Axis.Z)[-1]) as chop_sketch:

        RectangleRounded(
            height=CHOP_LENGTH,
            width=CHOP_WIDTH,
            radius=CHOP_CORNER_RADIUS,
            align=(Align.CENTER, Align.CENTER)
        )
    # # Extrude the bin to the specified height
    extrude(amount=-CHOP_HEIGHT, mode=Mode.SUBTRACT)


chop_part.label = "Chopping Block Bin"
show(chop_part)

+


# 5. Chopping Block Bin with scoops...

Chopping block bin has scoops in the right place and with some help from the lovely people on the CadQuery Discord server, it's now working.

In [47]:
with BuildPart() as chop_scoop_part:
    # Add the base
    add(BIN_BASE)

    # Add a sketch on top for the chop compartment
    with BuildSketch(chop_scoop_part.faces().sort_by(Axis.Z)[-1]):

        RectangleRounded(
            height=BASE_LENGTH * 42 * MM,
            width=BASE_WIDTH * 42 * MM,
            radius=BASE_CORNER_RADIUS,
            align=(Align.CENTER, Align.CENTER)
        )
        RectangleRounded(
            height=CHOP_LENGTH,
            width=CHOP_WIDTH,
            radius=CHOP_CORNER_RADIUS,
            mode=Mode.SUBTRACT,
            align=(Align.CENTER, Align.CENTER)
        )
    # Extrude the bin to the specified height
    extrude(amount=CHOP_HEIGHT)

    # Select the long sides of the chop block...
    with Locations(
        Plane(chop_scoop_part.faces().sort_by(Axis.X)[0]),
        Plane(chop_scoop_part.faces().sort_by(Axis.X)[-1]).rotated((0, 180, 180)),
    ) as long_sides:
        scoop_profile = ScoopProfile(
            height=CHOP_HEIGHT,
            length=CUTOUT_LENGTH,
            arc=CUTOUT_ARC,
            radius=CUTOUT_RADIUS,
            rotation=180.0,
            # align=(Align.CENTER, Align.CENTER),
            # align=(Align.CENTER, Align.MAX),
            # align=(Align.CENTER, Align.MIN),
        )
    extrude(to_extrude=scoop_profile, amount=-CUTOUT_DEPTH, mode=Mode.SUBTRACT)


chop_scoop_part.label = "Chopping block bin with scoops"
show(chop_scoop_part)
export_step(to_export=chop_scoop_part.part, file_path="chop_block.step")
export_stl(to_export=chop_scoop_part.part, file_path="chop_block.stl")


+


True

In [46]:
class ChopBin(BasePartObject):
    """Gridfinity Bin object with quirky compartment for storing IKEA chopping boards."""

    def __init__(
        self,
        height: float = 0,
        height_in_units: int = 0,
        rotation: RotationLike = (0, 0, 0),
        align: Align | tuple[Align, Align, Align] | None = None,
        mode: Mode = Mode.ADD,
    ):
        """Construct a custom bin object.

        Args:
            base (Part): Base object on which the bin is constructed.
            height (float, optional): Height of the bin in mm. Can't be used when height_in_units is
                defined.Defaults to 0.
            height_in_units (int, optional): Heigth defined by gridfinity units. Can't be used when
                height is defined. Defaults to 0.
            compartment (Compartment | None): Custom compartment of the bin, Defaults to None.
            rotation (RotationLike, optional): angles to rotate about axes. Defaults to (0, 0, 0).
            align (Union[Align, tuple[Align, Align, Align]], optional): align min, center, or max
            of object. Defaults to None.
            mode (Mode, optional): combination mode. Defaults to Mode.ADD.
        """
        if height and height_in_units:
            msg = "height or height_in_units can be defined, not both"
            raise ValueError(msg)
        if height_in_units:
            bin_height = height_in_units * 7
        else:
            bin_height = height

        with BuildPart() as bin:
            # Add the base
            add(
                BaseEqual(
                    grid_x=BASE_WIDTH,
                    grid_y=BASE_LENGTH,
                    rotation=rotation,
                    align=align,
                    mode=mode
                )
            )
            # Add a sketch on top for the chop compartment
            with BuildSketch(bin.faces().sort_by(Axis.Z)[-1]) as chop_sketch:

                RectangleRounded(
                    height=BASE_LENGTH * 42 * MM,
                    width=BASE_WIDTH * 42 * MM,
                    radius=BASE_CORNER_RADIUS * MM,
                    align=(Align.CENTER, Align.CENTER)
                )
                RectangleRounded(
                    height=CHOP_LENGTH * MM,
                    width=CHOP_WIDTH * MM,
                    radius=CHOP_CORNER_RADIUS * MM,
                    mode=Mode.SUBTRACT,
                    align=(Align.CENTER, Align.CENTER)
                )
            # Extrude the bin to the specified height
            extrude(to_extrude=chop_sketch.face(), amount=bin_height)
            # Add the cutout on the side for the chopping boards
            side_faces = [
                bin.faces().sort_by(Axis.X)[0],
                bin.faces().sort_by(Axis.X)[-1],
           ]
            with BuildSketch(side_faces):
                with BuildLine():
                    FilletPolyline(
                        (SIDE_HALF_LENGTH * MM, CHOP_HEIGHT * MM),
                        (SIDE_HALF_LENGTH * MM - SIDE_DOUBLE_LENGTH * MM, CHOP_HEIGHT * MM),
                        (SIDE_HALF_LENGTH * MM - SIDE_DOUBLE_LENGTH * MM, 0),
                        (0, 0),
                        radius=CUTOUT_RADIUS * MM,
                    )
                    Polyline(
                        (0, 0),
                        (0, CHOP_HEIGHT * MM),
                        (SIDE_HALF_LENGTH * MM, CHOP_HEIGHT * MM),
                    )

                # mirror(about=Plane.YZ)
            # Extrude the cutout from long side of bin
            extrude(amount=-2, mode=Mode.SUBTRACT)
            # # extrude(amount=BASE_WIDTH * 42 * -1 * MM, mode=Mode.SUBTRACT)

        super().__init__(bin.part, rotation, align, mode)

chop_block = ChopBin(
    height=CHOP_HEIGHT
)
show(chop_block)
# export_stl(chop_block, "chop_block.stl")

+
